# HireFlow 系统评估

运行全部 Cell 后，自动在 `reports/` 目录生成评估报告文件。

In [1]:
# ================================================================
# Cell 1: 初始化
# ================================================================
import sys, os, time, json
from datetime import datetime
import pytz

# 修复路径
project_root = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "evaluation" else os.getcwd()
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# 时间戳
sydney_tz = pytz.timezone("Australia/Sydney")
now = datetime.now(sydney_tz)
report_time = now.strftime("%I-%M-%p")    # 文件名用, 如 04-05-PM
display_time = now.strftime("%I:%M %p")    # 显示用, 如 04:05 PM
report_date = now.strftime("%Y-%m-%d")
report_filename = f"{report_date}-{report_time}.md"

# 确保 reports 目录存在
reports_dir = os.path.join(os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else os.getcwd(), "reports")
os.makedirs(reports_dir, exist_ok=True)

report_path = os.path.join(reports_dir, report_filename)

print(f"报告文件: reports/{report_filename}")
print(f"时间: {report_date} {display_time}")

报告文件: reports/2026-05-31-04-12-PM.md
时间: 2026-05-31 04:12 PM


In [2]:
# ================================================================
# Cell 2: 健康检查 (静默, 失败会抛异常中断)
# ================================================================
from app.utils.config import settings
from openai import OpenAI

# LLM
client = OpenAI(base_url=settings.llm.local_base_url, api_key=settings.llm.local_api_key)
models = [m.id for m in client.models.list().data]
assert settings.llm.local_model in models, f"LLM 模型 {settings.llm.local_model} 未找到"

# Embedding
client = OpenAI(base_url=settings.embedding.local_base_url, api_key=settings.embedding.local_api_key)
resp = client.embeddings.create(model=settings.embedding.local_model, input="test")
emb_dim = len(resp.data[0].embedding)

# PostgreSQL
from app.database.session import init_db, engine
init_db()

# Qdrant
from qdrant_client import QdrantClient
QdrantClient(url=settings.qdrant.url).get_collections()

print(f"健康检查通过 | LLM={settings.llm.local_model} | Embedding={settings.embedding.local_model}({emb_dim}维) | DB=PostgreSQL+Qdrant")
print(f"LLM模式={settings.llm.mode} | temperature={settings.llm.temperature}")

健康检查通过 | LLM=hermes-3-llama-3.1-8b | Embedding=text-embedding-qwen3-embedding-4b(2560维) | DB=PostgreSQL+Qdrant
LLM模式=local | temperature=0.1


In [3]:
# ================================================================
# Cell 3: 运行 Pipeline
# ================================================================
from app.agents.jd_agent import analyze_jd
from app.agents.resume_agent import batch_parse_resumes
from app.agents.match_agent import batch_match_candidates
from app.agents.ranking_agent import rank_candidates

# ---- 测试数据 ----
test_jd = """
岗位名称: Python 后端开发工程师
必备技能: Python, FastAPI, PostgreSQL, Docker, Git
加分技能: LangChain, RAG, Redis
岗位职责: 开发后端API, 数据库设计, 编写单元测试
学历要求: 计算机相关专业本科及以上
经验要求: 0-3年
"""

test_resumes = {
    "E001": "姓名: 张工\\n技能: Python, FastAPI, PostgreSQL, Docker, Git, Redis\\n项目: 电商API - FastAPI+PostgreSQL+Docker\\n教育: 2020-2024 北大 CS学士\\n经历: 2023某公司Python实习生",
    "E002": "姓名: 李工\\n技能: Python, Django, MySQL, Docker, Git\\n项目: 博客系统 - Django+MySQL\\n教育: 2019-2023 浙大 SE学士\\n经历: 2022某公司Django实习生",
    "E003": "姓名: 王工\\n技能: Python, FastAPI, PostgreSQL, LangChain, RAG, Docker, Git, Redis\\n项目: RAG问答系统 - FastAPI+LangChain+Qdrant\\n教育: 2021-2023 清华 AI硕士\\n经历: 2023某AI公司后端实习生",
}

print("运行 Pipeline...\n")
timeline = []
total_start = time.time()

# Step 1
print("  [1/4] JD解析...")
t0 = time.time()
jd_profile = await analyze_jd(test_jd)
t = time.time() - t0
timeline.append(("JD 解析", t))
jd_title = jd_profile.get("job_title", "?")
jd_skills = jd_profile.get("required_skills", [])
print(f"        {t:.1f}s | {jd_title} | 必备: {', '.join(jd_skills[:5])}")

# Step 2
print("  [2/4] 简历解析...")
t0 = time.time()
profiles = await batch_parse_resumes(test_resumes)
t = time.time() - t0
timeline.append(("简历解析", t))
ids = list(test_resumes.keys())
for i, p in enumerate(profiles):
    p["candidate_id"] = ids[i]
    print(f"        {p.get('name')}: {len(p.get('skills',[]))}技能 | {len(p.get('education',[]))}教育 | {len(p.get('projects',[]))}项目")
print(f"        {t:.1f}s ({len(profiles)}份)")

# Step 3
print("  [3/4] 匹配评分...")
t0 = time.time()
rubric = jd_profile.pop("rubric", None)
matches = await batch_match_candidates(jd_profile, profiles, rubric=rubric)
t = time.time() - t0
timeline.append(("匹配评分", t))
for m in matches:
    print(f"        {m.get('candidate_id')}: {m.get('total_score',0):.0f}分 → {m.get('recommendation','')}")
print(f"        {t:.1f}s ({len(matches)}人)")

# Step 4
print("  [4/4] 排序...")
t0 = time.time()
ranking = await rank_candidates(matches)
t = time.time() - t0
timeline.append(("排序", t))
print(f"        {t:.1f}s")

total_time = time.time() - total_start
print(f"\n  Pipeline 完成 | 总耗时: {total_time:.1f}s\n")

运行 Pipeline...

  [1/4] JD解析...


        3.7s | Python 后端开发工程师 | 必备: Python, FastAPI, PostgreSQL, Docker, Git
  [2/4] 简历解析...


        张工: 6技能 | 1教育 | 1项目
        李工: 5技能 | 1教育 | 1项目
        王工: 8技能 | 1教育 | 1项目
        20.6s (3份)
  [3/4] 匹配评分...


        E001: 85分 → Strong Match
        E002: 72分 → Medium Match
        E003: 87分 → Strong Match
        34.0s (3人)
  [4/4] 排序...


        2.2s

  Pipeline 完成 | 总耗时: 60.5s



In [4]:
# ================================================================
# Cell 4: 生成报告文件 + 打印摘要
# ================================================================
ranked = ranking.get("ranked_candidates", [])
summary = ranking.get("summary", {})
scores = [c.get("total_score", 0) for c in ranked]

# ---- 报告内容 ----
report = []
w = report.append  # 快捷方式

w(f"# HireFlow Pipeline 评估报告")
w(f"")
w(f"**日期:** {report_date}  **时间:** {display_time}  ")
w(f"**LLM:** {settings.llm.mode} ({settings.llm.local_model if settings.llm.mode == 'local' else settings.llm.cloud_model})  ")
w(f"**Embedding:** {settings.embedding.local_model} ({emb_dim}维)  ")
w(f"")

# ---- 性能 ----
w(f"## 一、Pipeline 性能")
w(f"")
w(f"| 步骤 | 耗时 | 占比 |")
w(f"|------|------|------|")
for name, t in timeline:
    pct = t / total_time * 100 if total_time > 0 else 0
    w(f"| {name} | {t:.1f}s | {pct:.0f}% |")
w(f"| **总计** | **{total_time:.1f}s** | **100%** |")
w(f"")
w(f"处理速度: {len(scores)/total_time:.2f} 候选人/秒  ")
w(f"")

# ---- 排序结果 ----
w(f"## 二、候选人排序")
w(f"")
w(f"| 排名 | 候选人 | 总分 | 技术 | 项目 | 经验 | 教育 | 领域 | 沟通 | 风险 | 等级 |")
w(f"|------|--------|------|------|------|------|------|------|------|------|------|")
for i, c in enumerate(ranked):
    score = c.get("total_score", 0)
    cid = c.get("candidate_id", "?")
    rec = c.get("recommendation", "")
    dims = c.get("dimension_scores", {})
    if isinstance(dims, dict):
        tech = dims.get("technical_skills", "-")
        proj = dims.get("project_relevance", "-")
        exp = dims.get("experience", "-")
        edu = dims.get("education", "-")
        domain = dims.get("domain_relevance", "-")
        comm = dims.get("communication", "-")
        risk = dims.get("risk_penalty", "-")
    else:
        tech = proj = exp = edu = domain = comm = risk = "-"
    w(f"| {i+1} | {cid} | {score:.0f} | {tech} | {proj} | {exp} | {edu} | {domain} | {comm} | {risk} | {rec} |")
w(f"")

# ---- 统计 ----
w(f"## 三、分数分布")
w(f"")
w(f"| 指标 | 值 |")
w(f"|------|----|")
w(f"| 候选人总数 | {len(scores)} |")
w(f"| Strong Match (≥80) | {summary.get('strong_match', 0)} |")
w(f"| Medium Match (65-79) | {summary.get('medium_match', 0)} |")
w(f"| Weak Match (50-64) | {summary.get('weak_match', 0)} |")
w(f"| Not Recommended (<50) | {summary.get('not_recommended', 0)} |")
if scores:
    w(f"| 最高分 | {max(scores):.0f} |")
    w(f"| 最低分 | {min(scores):.0f} |")
    w(f"| 平均分 | {sum(scores)/len(scores):.0f} |")
    w(f"| 分数极差 | {max(scores)-min(scores):.0f} |")
w(f"")

# ---- 观察 ----
w(f"## 四、观察与建议")
w(f"")
if scores and max(scores) >= 80:
    w(f"- 有 {summary.get('strong_match',0)} 位候选人达到 Strong Match 等级")
if scores and max(scores) - min(scores) > 20:
    w(f"- 分数极差 {max(scores)-min(scores):.0f} 分，候选人之间区分度明显")
elif scores and max(scores) - min(scores) < 10:
    w(f"- 分数极差仅 {max(scores)-min(scores):.0f} 分，候选人之间区分度不够")
if total_time > 60:
    w(f"- 总耗时 {total_time:.0f}s，主要瓶颈在{'匹配评分' if timeline[2][1] > timeline[1][1] else '简历解析'}阶段")
    w(f"- 优化方向: 切换云端模型(DeepSeek)可大幅缩短耗时")
if settings.llm.mode == "local":
    w(f"- 当前使用本地模型，免费但速度较慢。切换到 cloud 模式可获得更快、更稳定的评分")
w(f"")
w(f"---")
w(f"*报告由 HireFlow 评估系统自动生成*")

# ---- 写入文件 (用真实换行符) ----
with open(report_path, "w", encoding="utf-8") as f:
    f.write("\n".join(report))

# ---- 终端摘要 ----
print("=" * 50)
print(f"  报告已生成: reports/{report_filename}")
print("=" * 50)
print()
print(f"  Pipeline 总耗时: {total_time:.1f}s")
print(f"  候选人: {len(scores)} | Strong: {summary.get('strong_match',0)} | Medium: {summary.get('medium_match',0)} | Weak: {summary.get('weak_match',0)} | NR: {summary.get('not_recommended',0)}")
if scores:
    print(f"  分数范围: {min(scores):.0f} - {max(scores):.0f} | 平均: {sum(scores)/len(scores):.0f} | 极差: {max(scores)-min(scores):.0f}")
print()

# 打印报告内容预览
for line in report:
    print(line)

  报告已生成: reports/2026-05-31-04-12-PM.md

  Pipeline 总耗时: 60.5s
  候选人: 3 | Strong: 2 | Medium: 1 | Weak: 0 | NR: 0
  分数范围: 72 - 87 | 平均: 81 | 极差: 15

# HireFlow Pipeline 评估报告

**日期:** 2026-05-31  **时间:** 04:12 PM  
**LLM:** local (hermes-3-llama-3.1-8b)  
**Embedding:** text-embedding-qwen3-embedding-4b (2560维)  

## 一、Pipeline 性能

| 步骤 | 耗时 | 占比 |
|------|------|------|
| JD 解析 | 3.7s | 6% |
| 简历解析 | 20.6s | 34% |
| 匹配评分 | 34.0s | 56% |
| 排序 | 2.2s | 4% |
| **总计** | **60.5s** | **100%** |

处理速度: 0.05 候选人/秒  

## 二、候选人排序

| 排名 | 候选人 | 总分 | 技术 | 项目 | 经验 | 教育 | 领域 | 沟通 | 风险 | 等级 |
|------|--------|------|------|------|------|------|------|------|------|------|
| 1 | E003 | 87 | 30.0 | 20.0 | 15.0 | 10.0 | 8.0 | 5.0 | 0.0 | Strong Match |
| 2 | E001 | 85 | 25.0 | 18.0 | 15.0 | 10.0 | 8.0 | 5.0 | 0.0 | Strong Match |
| 3 | E002 | 72 | 20.0 | 15.0 | 12.0 | 8.0 | 7.0 | 4.0 | -3.0 | Medium Match |

## 三、分数分布

| 指标 | 值 |
|------|----|
| 候选人总数 | 3 |
| Strong Match (≥80) | 2 |
| Medium Match (65-

### 使用说明

**前置条件 (需要提前在终端启动):**
```bash
# 终端 1: 数据库
docker compose up -d postgres qdrant

# 终端 2 或桌面应用: LM Studio
# 加载 hermes-3-llama-3.1-8b + text-embedding-qwen3-embedding-4b
```

**运行 Notebook:**
```bash
conda activate hireflowagents
cd evaluation
jupyter notebook 系统评估报告.ipynb
# 或直接在 VS Code 中打开 .ipynb 文件，Run All
```

**报告文件:** 每次运行后在 `evaluation/reports/` 生成一个 `.md` 文件。

**云端模式:** 修改 `.env` 中 `LLM_MODE=cloud` 并填入 `LLM_CLOUD_API_KEY`